# 06. Thông tin bậc hai

Newton hội tụ sau một vòng trên hàm bậc hai. Điều đáng đo là **giá của vòng lặp đó**.
Chương 7 của báo cáo.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

In [2]:
import time
from scipy.linalg import cho_factor
from src.dataset import load_processed, SWEEP, FULL
from src.experiment import BUILDERS, ExperimentGroup, run_group
from src.figures import convergence_pair
obj, *_ = load_processed("../data/processed", SWEEP)
title, build = BUILDERS["newton-damping"]
recs = run_group(ExperimentGroup("newton-damping", title, build), obj, "../results/raw")
convergence_pair(recs, "newton-damping", title=title, out_dir="../results/figures")

newton-damping: loaded 4 runs from ../results/raw/newton-damping.json


{'iters': [PosixPath('../results/figures/newton-damping_iters.pdf'),
  PosixPath('../results/figures/newton-damping_iters.png')],
 'time': [PosixPath('../results/figures/newton-damping_time.pdf'),
  PosixPath('../results/figures/newton-damping_time.png')]}

## Tách chi phí một bước Newton

Con số đo được ở biến thể "Hessian dùng lại" **bỏ sót** phần dựng Hessian, vì `warm_up`
đã tính trước khi bấm đồng hồ. Đo riêng từng phần.

In [3]:
full, *_ = load_processed("../data/processed", FULL)
full.value_and_gradient(np.zeros(full.d))          # lam nong
t = time.perf_counter(); H = full.compute_hessian(); t_h = time.perf_counter() - t
t = time.perf_counter(); cho_factor(H, lower=True); t_c = time.perf_counter() - t
t = time.perf_counter(); full.gradient(np.zeros(full.d)); t_g = time.perf_counter() - t
pd.DataFrame([{"thành phần": "dựng X^T X", "ms": t_h*1e3, "bậc": "O(n d^2)"},
              {"thành phần": "Cholesky", "ms": t_c*1e3, "bậc": "O(d^3)"},
              {"thành phần": "một gradient", "ms": t_g*1e3, "bậc": "O(n d)"}]).round(2)

,thành phần,ms,bậc
0,dựng X^T X,121.61,O(n d^2)
1,Cholesky,0.27,O(d^3)
2,một gradient,78.89,O(n d)
